# Statevector quantum subspace expansion on H₂

## Imports

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from qibochem.driver import Molecule
from qibochem.ansatz.ucc import Ansatz_UCCSD
from qibochem.measurement.protocol import StateVectorProtocol
from qibochem.selected_ci import QSE_Computable, generate_singlet_singles, solve_generalised_eigeneqn

## 1. Build the molecule

Create a `Molecule` object for H₂ using Cartesian coordinates in
angstroms and the STO-3G basis set. Molecular geometries can also
be supplied through an XYZ file.

Call `run_pyscf()` to perform the Hartree–Fock calculation and
construct the molecular orbitals and integrals.

In [ ]:
h2 = Molecule([
    ("H", (0.0, 0.0, 0.0)),
    ("H", (0.0, 0.0, 0.735)),
], basis="sto-3g")
h2.run_pyscf()

print(f"HF energy: {h2.e_hf:.12f} Ha")
print(f"Electrons: {h2.nelec}; spin orbitals: {h2.nso}")

HF energy: -1.116998996754 Ha
Electrons: 2; spin orbitals: 4


## 2. Optimise the UCCSD reference state

Build a UCCSD ansatz and optimise its parameters with the
`L-BFGS-B` classical optimiser. `StateVectorProtocol()` evaluates
expectation values without measurement shot noise.

With `fast=True`, the optimisation applies parameterised Pauli
rotations directly to the statevector (instead of repeated evaluating the full VQE circuit). This is the recommended mode if hardware-noise is not relevant since many circuit evaluations are required during VQE optimisation.
After optimisation,`run_vqe()` returns the energy, optimised parameters, and a
Qibo circuit preparing the reference state.

In [ ]:
protocol = StateVectorProtocol()
ansatz = Ansatz_UCCSD(
    mol=h2, 
    ferm_qubit_map="jw",
    trotter_steps=1,
    include_hf=True,
    use_mp2_guess=True
)

vqe_energy, vqe_params, final_circuit = ansatz.run_vqe(
    protocol=protocol, 
    method="L-BFGS-B", 
    fast=True
)

print(f"Ansatz parameters: {len(ansatz.param_names)}")
print(f"Circuit gates: {len(final_circuit.queue)}")
print(f"VQE energy: {vqe_energy:.12f} Ha")
print(vqe_params)

[Qibo 0.3.2|INFO|2026-09-21 17:42:48]: Using numpy backend on /CPU:0


Ansatz parameters: 3
Circuit gates: 198
VQE energy: -1.137306035753 Ha
{'s0': np.float64(-8.213539269144507e-09), 's1': np.float64(-8.213539269144507e-09), 'd0': np.float64(0.11176849986109837)}


## 3. Construct and solve the QSE matrices

Create a `QSE_Computable` using the molecular Hamiltonian and
`generate_singlet_singles`. This generator constructs spin-adapted
one-body operators across the spatial orbitals, including
number operators.

- `observable=h2.hamiltonian("f")` selects the fermionic Hamiltonian.
- `ferm_qubit_map="jw"` selects the Jordan–Wigner mapping.
- `map_threshold=1e-12` controls the removal of small coefficients
  during observable construction.
- `spin_projection` is unused by this singlet generator. For the
  triplet generator, it selects the requested spin-projection
  component or components.

Evaluate the projected Hamiltonian matrix `H` and overlap matrix
`S` on the optimised reference circuit using `run_qse()`. Solve
the generalised eigenvalue problem

$$
Hc = ESc.
$$

The solver retains overlap eigenvectors with eigenvalues greater
than `1e-8`. It returns the QSE energies, their coefficient vectors,
and the retained subspace dimension. This overlap cutoff is
separate from `map_threshold`.

In [ ]:
qse = QSE_Computable(
    molecule=h2, 
    excitation_generator=generate_singlet_singles, 
    observable=h2.hamiltonian("f"),
    spin_projection=0,
    ferm_qubit_map="jw", 
    map_threshold=1e-12,
)

H, S = qse.run_qse(
    circuit=final_circuit, 
    protocol=protocol
)

qse_energies, qse_vectors, qse_rank = solve_generalised_eigeneqn(H, S, threshold=1e-8)

print("H:")
print(np.real_if_close(H))
print("S:")
print(np.real_if_close(S))
print(f"Retained QSE rank: {qse_rank}")
print(f"QSE correction from VQE: {qse_energies[0] - vqe_energy:.12e} Ha")

H:
[[-4.41241293e+00  3.61708812e-08 -3.25037479e-09 -8.02176567e-02]
 [ 3.61708812e-08 -4.04938503e-03  3.60791195e-02  3.64809873e-10]
 [-3.25037479e-09  3.60791195e-02 -3.21456927e-01  4.39656254e-09]
 [-8.02176567e-02  3.64809873e-10  4.39656254e-09  2.36240997e-02]]
S:
[[ 3.95023894e+00 -3.08169505e-08 -1.63245801e-08  1.22124533e-14]
 [-3.08169505e-08  2.48805316e-02 -2.21680000e-01  1.83220963e-09]
 [-1.63245801e-08 -2.21680000e-01  1.97511947e+00 -1.26601608e-08]
 [ 1.22124533e-14  1.83220963e-09 -1.26601608e-08  4.97610632e-02]]
Retained QSE rank: 3
QSE correction from VQE: 1.953992523340e-14 Ha


## 4. Evaluate spin expectations

Use `h2.s2_operator()` to construct the matrix of $\hat S^2$ in
the same QSE basis.

For each energy eigenvector `c` from the preceding calculation,
evaluate

$$
\langle \hat S^2 \rangle
= \frac{c^\dagger S^2_{\mathrm{QSE}} c}{c^\dagger S c}.
$$

Here, $S^2_{\mathrm{QSE}}$ denotes `S2_matrix`, rather than the
square of the overlap matrix. The table reports each QSE energy
and its spin expectation.

In [7]:
spin_qse = QSE_Computable(
    molecule=h2, 
    excitation_generator=generate_singlet_singles, 
    observable=h2.s2_operator(),
    spin_projection=0,
    ferm_qubit_map="jw", 
    map_threshold=1e-12,
)

S2_matrix, spin_overlap = spin_qse.run_qse(
    circuit=final_circuit, 
    protocol=protocol
)

spin_values = np.array([
    np.real(np.vdot(c, S2_matrix @ c) / np.vdot(c, S @ c))
    for c in qse_vectors.T
])

df_qse_results = pd.DataFrame({
    "root": [f"S{root}" for root in range(qse_rank)],
    "energy_Ha": qse_energies,
    "spin_squared": spin_values,
})
display(df_qse_results)

,root,energy_Ha,spin_squared
0,S0,-1.137306,1.001976e-14
1,S1,-0.162753,1.096691e-15
2,S2,0.495058,2.130213e-13
